In [12]:
import os
from dotenv import load_dotenv
from bardapi import Bard
import anthropic
from IPython.display import Markdown, display, update_display

import time
import google.generativeai as genai
import gradio as gr

In [13]:
# Load biến môi trường từ file .env
load_dotenv(override=True)

# Lấy API key
bard_api_key = os.getenv('BARD_API_KEY')

# Gán key và khởi tạo Bard nếu có
if bard_api_key:
    os.environ['_BARD_API_KEY'] = bard_api_key
    bard = Bard()
    print(f"Bard API Key loaded and begins with: {bard_api_key[:8]}")
else:
    print("Bard API Key not set")


genai.configure(api_key=bard_api_key)

Bard API Key loaded and begins with: AIzaSyB6


In [20]:
system_message = "You are a helpful assistant"

def chat(message, history):
    # Kết hợp system message, lịch sử và tin nhắn mới
    prompt = f"{system_message}\n"
    for msg in history:
        prompt += f"{msg['role'].capitalize()}: {msg['content']}\n"
    prompt += f"User: {message}\nAssistant:"

    # Tạo model và sinh nội dung dạng stream
    model = genai.GenerativeModel("gemini-1.5-pro")
    stream = model.generate_content(prompt, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response

# Giao diện ChatInterface sử dụng kiểu messages
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7878

To create a public link, set `share=True` in `launch()`.


In [32]:
system_message = "You are a helpful assistant"

def chat1(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # Chuyển danh sách messages thành một chuỗi prompt cho Gemini
    prompt = ""
    for msg in messages:
        role = msg["role"]
        if role == "system":
            prompt += f"[System]: {msg['content']}\n"
        elif role == "user":
            prompt += f"User: {msg['content']}\n"
        elif role == "assistant":
            prompt += f"Assistant: {msg['content']}\n"

    # Thêm dòng chờ Assistant trả lời
    prompt += "Assistant:"

    # Gọi Gemini và stream phản hồi
    model = genai.GenerativeModel("gemini-1.5-pro")
    stream = model.generate_content(prompt, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response

# Giao diện Gradio
gr.ChatInterface(fn=chat1, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7884

To create a public link, set `share=True` in `launch()`.
